In [45]:
import pandas as pd

In [46]:
dataset = pd.read_csv("DATASET_ANEMIA.csv")

In [47]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113537 entries, 0 to 113536
Data columns (total 29 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   id_mujer                 113537 non-null  object
 1   orden_nacimiento         113537 non-null  int64 
 2   linea_nino_hogar         113537 non-null  int64 
 3   peso_nacer_g             113537 non-null  int64 
 4   anemia_materna_embarazo  113537 non-null  object
 5   desparasitacion          113537 non-null  int64 
 6   diarrea_reciente         113537 non-null  int64 
 7   fiebre_reciente          113537 non-null  int64 
 8   nivel_anemia             113537 non-null  int64 
 9   edad_meses               113537 non-null  int64 
 10  peso_nino_kg             113537 non-null  int64 
 11  talla_nino_cm            113537 non-null  int64 
 12  z_talla_edad             113537 non-null  int64 
 13  z_peso_edad              113537 non-null  int64 
 14  z_peso_talla        

In [48]:
dataset.describe()

,orden_nacimiento,linea_nino_hogar,peso_nacer_g,desparasitacion,diarrea_reciente,fiebre_reciente,nivel_anemia,edad_meses,peso_nino_kg,talla_nino_cm,...,anios_educacion_madre,quintil_riqueza,area_residencia,fuente_agua_hogar,ubigeo,altitud_residencia_m,sexo_nino,peso_madre_kg,talla_madre_cm,anemia_materna_actual
count,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,...,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000,113537.000000
mean,1.118411,4.419440,3454.649911,0.374336,0.232171,0.187754,4.063891,30.397773,140.966989,997.640549,...,10.642945,2.463206,1.304694,26.672036,133726.333266,1253.514308,1.489884,714.437355,1583.083224,3.783322
std,0.338017,1.584693,1242.982145,0.483953,0.650156,0.414415,1.707422,16.944197,108.218336,1081.933142,...,3.710211,1.298491,0.460280,26.719418,69634.519515,1411.123223,0.499900,772.965159,697.639142,0.757819
min,1.000000,2.000000,500.000000,0.000000,0.000000,0.000000,1.000000,0.000000,17.000000,367.000000,...,0.000000,1.000000,1.000000,11.000000,10101.000000,1.000000,1.000000,204.000000,1049.000000,1.000000
25%,1.000000,3.000000,2965.000000,0.000000,0.000000,0.000000,3.000000,16.000000,100.000000,769.000000,...,9.000000,1.000000,1.000000,11.000000,70107.000000,114.000000,1.000000,564.000000,1488.000000,4.000000
50%,1.000000,4.000000,3300.000000,0.000000,0.000000,0.000000,4.000000,31.000000,129.000000,891.000000,...,11.000000,2.000000,1.000000,11.000000,140306.000000,408.000000,1.000000,640.000000,1525.000000,4.000000
75%,1.000000,5.000000,3645.000000,1.000000,0.000000,0.000000,4.000000,45.000000,157.000000,984.000000,...,14.000000,3.000000,2.000000,41.000000,190112.000000,2680.000000,2.000000,726.000000,1563.000000,4.000000
max,4.000000,25.000000,9998.000000,1.000000,8.000000,8.000000,9.000000,59.000000,999.000000,9999.000000,...,18.000000,5.000000,2.000000,97.000000,250401.000000,5053.000000,2.000000,9999.000000,9999.000000,9.000000


In [49]:
#quitar los identificadores
dataset = dataset.drop(columns=['orden_nacimiento','linea_nino_hogar','id_mujer'])

In [50]:
#ELIMINAR VALORES IRREGULARES DEL DATASET (ORIGINAL)
#del niño
dataset = dataset[dataset['peso_nino_kg'] != 999]
dataset = dataset[dataset['talla_nino_cm'] != 9999]
dataset = dataset[dataset['peso_nacer_g'] != 9996]
dataset = dataset[dataset['peso_nacer_g'] != 9998]
dataset = dataset[~dataset['z_talla_edad'].isin([9998, 9997, 9996])]
dataset = dataset[~dataset['z_peso_edad'].isin([9998, 9997, 9996])]
dataset = dataset[~dataset['z_peso_talla'].isin([9998, 9997, 9996])]
dataset = dataset[~dataset['z_imc_edad'].isin([9998, 9997, 9996])]
dataset = dataset[dataset['nivel_anemia'] != 9]
dataset = dataset[dataset['diarrea_reciente'] != 8]
dataset = dataset[dataset['fiebre_reciente'] != 8]

#de la madre
dataset = dataset[dataset['talla_madre_cm'] != 9999]
dataset = dataset[dataset['peso_madre_kg'] != 9999]
dataset = dataset[dataset['anemia_materna_actual'] != 9]

#reemplazar valores faltantes:
dataset['anemia_materna_embarazo'] = dataset['anemia_materna_embarazo'].replace(' ', '8')

#balancear los registros (nivel de anemia -> anemia (si/no))
dataset['nivel_anemia'] = dataset['nivel_anemia'].replace(4, 0)
dataset['nivel_anemia'] = dataset['nivel_anemia'].replace(3, 1)
dataset['nivel_anemia'] = dataset['nivel_anemia'].replace(2, 1)

In [51]:
df = dataset
#AGREGAR COLUMNA DE DEPARTAMENTOS:
df['ubigeo_str'] = df['ubigeo'].astype(str).str.zfill(6)
df['dep_codigo'] = df['ubigeo_str'].str[:2]

# Diccionario completo de códigos UBIGEO de 2 dígitos por departamento
df_departamentos = pd.DataFrame({
    'dep_codigo': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
                 '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25'],
    'departamento': ['Amazonas', 'Áncash', 'Apurímac', 'Arequipa', 'Ayacucho', 'Cajamarca',
                     'Callao', 'Cusco', 'Huancavelica', 'Huánuco', 'Ica', 'Junín',
                     'La Libertad', 'Lambayeque', 'Lima', 'Loreto', 'Madre de Dios',
                     'Moquegua', 'Pasco', 'Piura', 'Puno', 'San Martín', 'Tacna',
                     'Tumbes', 'Ucayali'],
    'nivel_pobreza': [ 3, 3, 3, 2, 5, 5, 4, 3, 5, 5, 1, 2, 4, 2, 4, 5, 1, 1, 5, 4, 5, 3, 3, 4, 3]
})

df = pd.merge(df, df_departamentos, on='dep_codigo', how='left')

#cambiar tipo de dato object a int
df['anemia_materna_embarazo'] = pd.to_numeric(df['anemia_materna_embarazo'], errors='coerce').astype('Int64')

In [52]:
df.describe()

,peso_nacer_g,anemia_materna_embarazo,desparasitacion,diarrea_reciente,fiebre_reciente,nivel_anemia,edad_meses,peso_nino_kg,talla_nino_cm,z_talla_edad,...,quintil_riqueza,area_residencia,fuente_agua_hogar,ubigeo,altitud_residencia_m,sexo_nino,peso_madre_kg,talla_madre_cm,anemia_materna_actual,nivel_pobreza
count,99472.000000,99472.0,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,...,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000,99472.000000
mean,3262.478939,2.711316,0.406758,0.235544,0.189269,0.338859,32.543429,134.185590,891.431478,-80.171465,...,2.483804,1.294234,26.535085,133121.542595,1260.758696,1.491033,654.310520,1526.244360,3.742018,3.444165
std,543.124178,2.330582,0.491231,0.644679,0.391724,0.473324,15.656212,35.633854,119.639922,101.914907,...,1.291029,0.455700,26.754172,69419.787962,1413.979796,0.499922,126.157346,55.469082,0.526367,1.268500
min,500.000000,1.0,0.000000,0.000000,0.000000,0.000000,6.000000,43.000000,552.000000,-587.000000,...,1.000000,1.000000,11.000000,10101.000000,1.000000,1.000000,204.000000,1049.000000,1.000000,1.000000
25%,2950.000000,2.0,0.000000,0.000000,0.000000,0.000000,19.000000,107.000000,799.000000,-148.000000,...,1.000000,1.000000,11.000000,70106.000000,113.000000,1.000000,566.000000,1489.000000,4.000000,3.000000
50%,3280.000000,2.0,0.000000,0.000000,0.000000,0.000000,33.000000,132.000000,904.000000,-81.000000,...,2.000000,1.000000,11.000000,140301.000000,418.000000,1.000000,641.000000,1525.000000,4.000000,4.000000
75%,3600.000000,2.0,1.000000,0.000000,0.000000,1.000000,46.000000,158.000000,987.000000,-13.000000,...,3.000000,2.000000,41.000000,190109.000000,2686.000000,2.000000,727.000000,1562.000000,4.000000,4.000000
max,6000.000000,8.0,1.000000,2.000000,1.000000,1.000000,59.000000,346.000000,1198.000000,594.000000,...,5.000000,2.000000,97.000000,250401.000000,5053.000000,2.000000,1560.000000,1885.000000,4.000000,5.000000


In [53]:
df = df.drop(columns=['ubigeo','ubigeo_str'])

In [54]:
df['dep_codigo'] = pd.to_numeric(df['dep_codigo'], errors='coerce').astype('Int64')

In [55]:
df

,peso_nacer_g,anemia_materna_embarazo,desparasitacion,diarrea_reciente,fiebre_reciente,nivel_anemia,edad_meses,peso_nino_kg,talla_nino_cm,z_talla_edad,...,area_residencia,fuente_agua_hogar,altitud_residencia_m,sexo_nino,peso_madre_kg,talla_madre_cm,anemia_materna_actual,dep_codigo,departamento,nivel_pobreza
0,3216,2,0,0,0,0,40,147,955,-62,...,1,11,2347,2,562,1530,4,1,Amazonas,3
1,2930,1,1,0,0,0,53,169,1023,-98,...,1,11,2347,1,474,1457,4,1,Amazonas,3
2,2730,2,0,0,0,1,8,103,720,62,...,1,11,2347,1,602,1522,4,1,Amazonas,3
3,3340,1,0,0,0,0,22,115,842,-26,...,1,11,2416,2,505,1572,3,1,Amazonas,3
4,3170,2,1,0,0,0,21,123,841,-46,...,1,11,2416,1,573,1495,4,1,Amazonas,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99467,2595,2,0,0,0,0,29,120,866,-156,...,2,71,191,1,557,1509,4,25,Ucayali,3
99468,3710,2,1,0,0,0,35,124,884,-202,...,2,71,191,1,623,1455,4,25,Ucayali,3
99469,3720,1,1,0,0,0,55,175,1069,-16,...,2,71,191,1,718,1493,4,25,Ucayali,3
99470,3265,1,1,0,0,0,22,98,766,-274,...,2,11,191,2,642,1483,4,25,Ucayali,3


In [56]:
#convertir peso al nacer (g -> kg)
df['peso_nacer_kg'] = df['peso_nacer_g'] / 1000
df = df.drop(columns=['peso_nacer_g'])

#cambiar valores de anemia_materna_embarazo (-1: no se hizo prueba, 0: no tenía anemia, 1: tenia anemia)
df['anemia_materna_embarazo'] = df['anemia_materna_embarazo'].replace(2, 0)
df['anemia_materna_embarazo'] = df['anemia_materna_embarazo'].replace(1, 1)
df['anemia_materna_embarazo'] = df['anemia_materna_embarazo'].replace(8, -1)

#arreglar el peso: 103 -> 10.3 cm
df['peso_nino_kg'] = df['peso_nino_kg'] / 10

#arreglar la talla: 720 -> 72.0 cm
df['talla_nino_cm'] = df['talla_nino_cm'] / 10

#arreglar los indicadores:
df['z_talla_edad'] = df['z_talla_edad'] / 100
df['z_peso_edad'] = df['z_peso_edad'] / 100
df['z_peso_talla'] = df['z_peso_talla'] / 100
df['z_imc_edad'] = df['z_imc_edad'] / 100

#arreglar el peso de la madre: 602 -> 60.2 kg
df['peso_madre_kg'] = df['peso_madre_kg'] / 10

#arreglar la talla de la madre: 1522 -> 152.2 cm
df['talla_madre_cm'] = df['talla_madre_cm'] / 10

#hacer la talla de la madre en metros:
df['talla_madre_m'] = df['talla_madre_cm'] / 100

#arreglar la anemia de la madre (binario)
df['anemia_materna_actual'] = df['anemia_materna_actual'].replace(4, 0)
df['anemia_materna_actual'] = df['anemia_materna_actual'].replace(3, 1)
df['anemia_materna_actual'] = df['anemia_materna_actual'].replace(2, 1)
df['anemia_materna_actual'] = df['anemia_materna_actual'].replace(1, 1)

In [57]:
df = df.drop(columns=['fuente_agua_hogar','talla_madre_cm','dep_codigo'])

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99472 entries, 0 to 99471
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   anemia_materna_embarazo  99472 non-null  Int64  
 1   desparasitacion          99472 non-null  int64  
 2   diarrea_reciente         99472 non-null  int64  
 3   fiebre_reciente          99472 non-null  int64  
 4   nivel_anemia             99472 non-null  int64  
 5   edad_meses               99472 non-null  int64  
 6   peso_nino_kg             99472 non-null  float64
 7   talla_nino_cm            99472 non-null  float64
 8   z_talla_edad             99472 non-null  float64
 9   z_peso_edad              99472 non-null  float64
 10  z_peso_talla             99472 non-null  float64
 11  z_imc_edad               99472 non-null  float64
 12  anio_encuesta            99472 non-null  int64  
 13  edad_madre               99472 non-null  int64  
 14  nivel_educativo_madre 

In [59]:
df.rename(columns={'nivel_anemia':'ANEMIA'}, inplace = True)

In [60]:
from google.colab import files

# 1. Crear y guardar el CSV en el entorno virtual de Colab
df.to_csv('DATASET_LIMPIO_ANEMIA.csv', index=False, encoding='utf-8-sig')

# 2. Descargar el archivo a tu computadora
files.download('DATASET_LIMPIO_ANEMIA.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>